# 📘 Notebook 11: Neptune Analytics & Gremlin Query Language

## 🎯 Learning Objectives

By the end of this notebook, you will:
* Master **Gremlin traversal queries** for graph traversal at scale
* Understand **Neptune architecture** and how it differs from Neo4j
* Learn to build **fraud detection pipelines** with Neptune + SageMaker
* Understand **graph analytics algorithms** built into Neptune
* Compare **Gremlin vs Cypher** query approaches

---

## ⚡ Quick Context: Neptune vs Neo4j for Production

| Aspect | Neo4j | Neptune |
| --- | --- | --- |
| **Deployment** | Self-hosted or managed | AWS-native managed service |
| **Query Language** | Cypher (declarative) | Gremlin (imperative traversal) |
| **Scale** | ~100M-1B nodes | 1B+ nodes (distributed) |
| **Graph Algorithms** | APOC/GDS libraries | Built-in optimized algorithms |
| **Real-time Speed** | Milliseconds | Milliseconds (with caching) |
| **Batch Processing** | Per-query | Neptune Analytics (parallel jobs) |
| **ML Integration** | Custom | Direct SageMaker integration |
| **Cost Model** | Per-node licensing | Pay per query/compute |

**Key Advantage:** Neptune = production-scale, enterprise-grade graph analytics

---

## 🏗️ Neptune Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                    APPLICATION LAYER                     │
│  (Web App, Mobile, API, Real-time Dashboards)           │
└──────────────────┬──────────────────────────────────────┘
                   │
┌──────────────────┴──────────────────────────────────────┐
│            QUERY EXECUTION LAYER                         │
│  ┌─────────────────────────────────────────────────────┐ │
│  │  Gremlin / SPARQL Query Engine                       │ │
│  │  (Traversal execution, optimization)                │ │
│  └─────────────────────────────────────────────────────┘ │
└──────────────────┬──────────────────────────────────────┘
                   │
┌──────────────────┴──────────────────────────────────────┐
│          GRAPH ANALYTICS LAYER (Neptune Analytics)       │
│  ┌─────────────────────────────────────────────────────┐ │
│  │ Centrality  │ Communities  │ Paths  │ Similarity   │ │
│  │ (PageRank)  │ (Louvain)     │ (BFS) │ (Node2Vec)   │ │
│  └─────────────────────────────────────────────────────┘ │
└──────────────────┬──────────────────────────────────────┘
                   │
┌──────────────────┴──────────────────────────────────────┐
│            DATA STORAGE & INDEXING LAYER                 │
│  ┌──────────┬──────────┬──────────┬──────────────────┐  │
│  │ Vertices │ Edges    │ Indices  │ Property Store   │  │
│  │ Storage  │ Storage  │ (B-tree) │ (Key-value)      │  │
│  └──────────┴──────────┴──────────┴──────────────────┘  │
└──────────────────┬──────────────────────────────────────┘
                   │
┌──────────────────┴──────────────────────────────────────┐
│            AWS INTEGRATION LAYER                         │
│  ┌──────────┬──────────┬──────────┬──────────────────┐  │
│  │ S3 (data)│ Lambda   │SageMaker │ CloudWatch       │  │
│  │ (import) │(real-time)│(ML)     │ (monitoring)     │  │
│  └──────────┴──────────┴──────────┴──────────────────┘  │
└─────────────────────────────────────────────────────────┘
```

**Data Flow:**
1. Data ingestion → S3 → Neptune (bulk load)
2. Real-time updates → Lambda → Neptune (immediate)
3. Queries → Gremlin Engine → Results (cached)
4. Analytics → Neptune Analytics → Features → SageMaker
5. Monitoring → CloudWatch → Alerts

---

## 🔄 Gremlin Traversal Fundamentals

**Gremlin** is an imperative graph traversal language (vs Cypher's declarative approach):
- You specify **HOW** to traverse (step-by-step)
- Vs Cypher: you specify **WHAT** you want to find

**Basic Structure:**
```
g.V()              # Start: all vertices
  .has()           # Filter step
  .out()           # Traversal step
  .values()        # Project step
```

Each step in the chain is a **traverser** - carries data through the pipeline.

---

## 🚀 Running Example: Same Fraud Detection Network

We'll use the **identical transaction network** from Notebook 10 to compare Gremlin vs Cypher:

```
Accounts: Alice, Bob, Charlie, David, Eve, Frank
Transactions: 8 TRANSFERS relationships with amounts and dates
Cycle: Alice → Bob → Frank → Alice ($6950 total)
Hub: Charlie (receives from 3 sources, sends to 2)

Example query formats in Gremlin:
1. Simple traversal
2. Filtering 
3. Multi-hop paths
4. Aggregation
5. Pattern detection
6. Hub analysis
7. Risk scoring
```

Each Gremlin query will be shown in **multiple formats** for clarity.

In [ ]:
# 🧩 Part 1: Neptune Setup & Gremlin Basics

import json
from datetime import datetime

print("=" * 70)
print("NEPTUNE & GREMLIN QUERY TUTORIAL")
print("=" * 70)

print("\n📊 FRAUD TRANSACTION NETWORK (Same as Notebook 10):")
print("-" * 70)
print("""
Accounts:
  - alice: balance=$10,000, risk=LOW
  - bob: balance=$5,000, risk=MEDIUM
  - charlie: balance=$2,000, risk=HIGH
  - david: balance=$8,000, risk=MEDIUM
  - eve: balance=$1,000, risk=HIGH
  - frank: balance=$3,000, risk=MEDIUM

Transactions (TRANSFERS):
  1. alice → bob: $5,000 (2024-01-15)
  2. bob → charlie: $2,000 (2024-01-20)
  3. charlie → david: $1,500 (2024-02-01)
  4. alice → eve: $500 (2024-02-10)
  5. bob → frank: $1,000 (2024-02-15)
  6. frank → alice: $950 (2024-03-01) ⚠️ CYCLE
  7. david → charlie: $3,000 (2024-03-15)
  8. charlie → eve: $2,000 (2024-04-01)

Key Patterns:
  - Cycle: alice → bob → frank → alice
  - Hub: charlie (5 total connections)
  - Chain: alice → bob → charlie → david (4-hop)
""")

print("\n" + "=" * 70)
print("GREMLIN SYNTAX FUNDAMENTALS")
print("=" * 70)

print("\n1️⃣ TRAVERSERS (Core Concept)")
print("-" * 70)
print("""
In Gremlin, everything flows through TRAVERSERS:
  - Each step produces 0 or more traversers
  - Traversers carry: (vertex/edge/value, path, metadata)
  - Chain steps together for complex traversals

Example: g.V().has('name','Alice').out('TRANSFERS')
  Step 1: g.V()           → 6 traversers (all vertices)
  Step 2: .has()          → 1 traverser (alice)
  Step 3: .out()          → 2 traversers (bob, eve)
""")

print("\n2️⃣ KEY GREMLIN METHODS")
print("-" * 70)
print("""
Traversal Methods:
  - V()          → All vertices
  - E()          → All edges
  - has()        → Filter by property
  - out()        → Outgoing edges (for directed)
  - in()         → Incoming edges
  - both()       → Any direction
  - property()   → Get/set properties

Transformation:
  - values()     → Extract property values
  - map()        → Transform traverser
  - select()     → Choose which columns
  - path()       → Capture path taken

Aggregation:
  - count()      → Count items
  - sum()        → Total values
  - group()      → Group by key
  - dedup()      → Remove duplicates

Control Flow:
  - filter()     → Conditional
  - where()      → Condition check
  - limit()      → Limit results
  - order()      → Sort
""")

print("\n3️⃣ GREMLIN vs CYPHER COMPARISON")
print("-" * 70)

comparison_queries = {
    "Task": "Find accounts Alice sends money to",
    "Cypher": """
    MATCH (a:Account {name: 'Alice'})-[:TRANSFERS]->(b:Account)
    RETURN b.name
    """,
    "Gremlin": """
    g.V().has('Account', 'name', 'Alice')
      .out('TRANSFERS')
      .values('name')
    """,
}

print("\n📌 Example: Find who Alice sends money to")
print(f"\nCypher (Declarative - WHAT to get):")
print(comparison_queries["Cypher"])
print(f"\nGremlin (Imperative - HOW to traverse):")
print(comparison_queries["Gremlin"])

print("\n💡 Key Differences:")
print("""
  Cypher:
    - Pattern-based: (node)-[:rel]->(node)
    - Declarative: Specify desired results
    - Single query string
    - Optimizer chooses execution plan
    
  Gremlin:
    - Step-by-step traversal
    - Imperative: Specify traversal steps
    - Method chaining (fluent API)
    - You control execution path
    - More verbose but more flexible
""")

print("\n✓ Setup complete. Ready for Gremlin queries...")


---

## 📝 Gremlin Query Type 1: Data Creation & Basic Traversal

### Setup: Bulk Loading Data to Neptune

In Neptune, you load data via Gremlin. Here's how to create our fraud network:

**Format: Gremlin Groovy (server-side execution)**

```groovy
// Add vertices (accounts)
account = graph.addVertex(label: 'Account')
account.property('id', 'alice')
account.property('name', 'Alice')
account.property('balance', 10000)
account.property('risk', 'LOW')

// Repeat for bob, charlie, david, eve, frank...

// Add edges (transactions)
alice = g.V().has('id', 'alice').next()
bob = g.V().has('id', 'bob').next()
edge = alice.addEdge('TRANSFERS', bob)
edge.property('amount', 5000)
edge.property('date', '2024-01-15')
edge.property('txn_id', 'txn001')
```

**In Production:** Use bulk loading from S3 instead of per-record insertion

In [ ]:
# 🔹 Gremlin Query Type 1: Simple Traversal (Direct Connections)

print("\n" + "=" * 70)
print("GREMLIN QUERY TYPE 1: SIMPLE TRAVERSAL - WHO DID ALICE SEND TO?")
print("=" * 70)

gremlin_query_1 = """
// Format 1: Step-by-step (readable)
g.V()
  .has('Account', 'id', 'alice')
  .out('TRANSFERS')
  .values('name')

// Format 2: Compact (production)
g.V().has('id','alice').out('TRANSFERS').values('name')

// Format 3: With edge properties
g.V().has('id','alice').outE('TRANSFERS').as('edge')
  .inV().as('recipient')
  .select('edge','recipient')
  .by(valueMap())

// Format 4: Include amounts
g.V().has('id','alice')
  .outE('TRANSFERS')
  .project('receiver', 'amount', 'date')
  .by(inV().values('name'))
  .by(values('amount'))
  .by(values('date'))
"""

print("\nGremlin Query (Multiple Formats):")
print("-" * 70)
print(gremlin_query_1)

print("\nExpected Result (Format 1 & 2):")
print("""
bob
eve
""")

print("\nExpected Result (Format 3 - With Edge Details):")
print("""
{
  "edge": {
    "id": "xyz123",
    "label": "TRANSFERS",
    "amount": 5000,
    "date": "2024-01-15"
  },
  "recipient": {
    "id": "bob_id",
    "name": "Bob",
    "balance": 5000
  }
}
{
  "edge": {
    "amount": 500,
    "date": "2024-02-10"
  },
  "recipient": {
    "name": "Eve"
  }
}
""")

print("\nExpected Result (Format 4 - Projected):")
print("""
[{"receiver": "Bob", "amount": 5000, "date": "2024-01-15"},
 {"receiver": "Eve", "amount": 500, "date": "2024-02-10"}]
""")

print("\n💡 Gremlin Concepts Explained:")
print("""
  g                    → Traversal source (entry point)
  .V()                 → All vertices
  .has('id','alice')   → Filter by property (label optional)
  .out('TRANSFERS')    → Outgoing TRANSFERS edges
  .values('name')      → Extract 'name' property
  
  outE()               → Outgoing edges (not vertices)
  inV()                → Destination vertex of edge
  as()                 → Name/alias for use in select()
  project()            → Select specific columns
  by()                 → Apply function to each step
  valueMap()           → Get all properties as map
""")

print("\n🔄 Comparison with Cypher:")
print("""
Cypher:
  MATCH (a:Account {id: 'alice'})-[t:TRANSFERS]->(b:Account)
  RETURN b.name, t.amount, t.date

Gremlin:
  g.V().has('id','alice')
    .outE('TRANSFERS')
    .project('name','amount','date')
    .by(inV().values('name'))
    .by(values('amount'))
    .by(values('date'))

Key difference: Cypher gets nodes/edges/properties in one pattern
              Gremlin steps through each transformation
""")

# Variation 1: Bidirectional traversal
print("\n\n" + "-" * 70)
print("VARIATION 1: Bidirectional (Both Sending AND Receiving)")
print("-" * 70)

gremlin_bidir = """
g.V().has('id','alice')
  .bothE('TRANSFERS')
  .project('direction','otherAccount','amount')
  .by(label)
  .by(bothV().where(neq('alice')).values('name'))
  .by(values('amount'))
  .order().by(select('amount'), decr)
"""

print("\nGremlin Query:")
print(gremlin_bidir)

print("\nExpected Result:")
print("""
{"direction":"TRANSFERS","otherAccount":"Bob","amount":5000}
{"direction":"TRANSFERS","otherAccount":"Frank","amount":950}
{"direction":"TRANSFERS","otherAccount":"Eve","amount":500}
""")

print("\n💡 New Concepts:")
print("""
  .bothE()     → Edges in either direction (in OR out)
  .bothV()     → Both vertices of edge
  .neq()       → Not equal filter
  .order()     → Sort results
  .by()        → Sort key and direction
  decr         → Descending order (incr = ascending)
""")

# Variation 2: Filter by transaction amount
print("\n\n" + "-" * 70)
print("VARIATION 2: High-Value Transactions Only (> $1000)")
print("-" * 70)

gremlin_filtered = """
g.V()
  .outE('TRANSFERS')
  .has('amount', gt(1000))
  .project('sender','receiver','amount')
  .by(outV().values('name'))
  .by(inV().values('name'))
  .by(values('amount'))
  .order().by(select('amount'), decr)
"""

print("\nGremlin Query:")
print(gremlin_filtered)

print("\nExpected Result:")
print("""
{"sender":"Alice","receiver":"Bob","amount":5000}
{"sender":"David","receiver":"Charlie","amount":3000}
{"sender":"Bob","receiver":"Charlie","amount":2000}
{"sender":"Charlie","receiver":"Eve","amount":2000}
{"sender":"Charlie","receiver":"David","amount":1500}
""")

print("\n💡 Key Points:")
print("""
  .has('amount', gt(1000))  → Has predicate (gt = greater than)
                             Also: lt, gte, lte, eq, neq
  
  Other common predicates:
    .has('property', eq(value))
    .has('property', inside(min, max))
    .has('property', startingWith('prefix'))
    .hasLabel('Account')     → Filter by vertex label
""")

print("\n✓ Query Type 1 covered: Simple traversal and filtering")

---

## 🔗 Gremlin Query Type 2: Multi-Hop Traversal (Repeating Patterns)

### Use Case: "Trace money flow through multiple hops"

**Gremlin Syntax for Repetition:**
- `.repeat()` - Repeat a pattern
- `.times(n)` - Exactly n times
- `.until(condition)` - Until condition met
- `.emit()` - Emit intermediate results

**Difference from Cypher:**
- Cypher: `*1..3` in relationship pattern
- Gremlin: Explicit `.repeat()` with loop control

This is where Gremlin's imperative nature shines for complex patterns.

In [ ]:
# 🔹 Gremlin Query Type 2: Multi-Hop Traversal with .repeat()

print("\n" + "=" * 70)
print("GREMLIN QUERY TYPE 2: MULTI-HOP TRAVERSAL")
print("=" * 70)

print("\nKey Concept: .repeat() loop with traversal steps")
print("-" * 70)

# Query 2.1: Exactly 2 hops
print("\n📌 Query 2.1: Exactly 2-Hop Paths from Alice")
print("-" * 70)

gremlin_2hop = """
// Find all accounts reachable from Alice in exactly 2 steps
g.V().has('id', 'alice')
  .repeat(out('TRANSFERS'))
  .times(2)
  .dedup()
  .values('name')

// With path tracking
g.V().has('id', 'alice')
  .repeat(out('TRANSFERS'))
  .times(2)
  .path()
  .by(values('name'))
"""

print("Gremlin Query:")
print(gremlin_2hop)

print("\nExpected Result (Direct):")
print("""
Charlie
Frank
""")

print("\nExpected Result (With Paths):")
print("""
[Alice, Bob, Charlie]
[Alice, Bob, Frank]
""")

print("\n💡 Explanation:")
print("""
  .repeat(out('TRANSFERS'))  → Loop: follow outgoing TRANSFERS
  .times(2)                  → Loop exactly 2 times
  .dedup()                   → Remove duplicates (Charlie via 2 paths)
  .path()                    → Capture the path taken
  .by(values('name'))        → Extract name for each vertex in path
""")

# Query 2.2: Variable hops with until
print("\n\n" + "-" * 70)
print("Query 2.2: Variable Hops (1-3) - Who's Reachable from Alice?")
print("-" * 70)

gremlin_varihop = """
// Find all accounts reachable in 1-3 hops
g.V().has('id', 'alice')
  .repeat(out('TRANSFERS'))
  .times(1)
  .until(loops().is(gte(3)))
  .dedup()
  .values('name')

// Better: With until condition
g.V().has('id', 'alice')
  .repeat(out('TRANSFERS')).until(loops().is(gte(3)))
  .dedup()
  .values('name')
"""

print("Gremlin Query:")
print(gremlin_varihop)

print("\nExpected Result:")
print("""
Bob
Charlie
David
Eve
Frank
""")

print("\n💡 Loop Control Functions:")
print("""
  .loops()           → Current loop iteration count
  .loops().is(gte(3))  → Continue until loops >= 3
  .loops().is(eq(2))   → Emit at exactly 2 iterations
  
  Other conditions:
    .loops().is(lt(3))     → Less than 3
    .loops().is(between(1,3))  → Between 1 and 3
""")

# Query 2.3: Emit intermediate results
print("\n\n" + "-" * 70)
print("Query 2.3: Emit All Intermediate Results (Alice & Her Reach)")
print("-" * 70)

gremlin_emit = """
// Emit every step (including starting node)
g.V().has('id', 'alice')
  .emit()
  .repeat(out('TRANSFERS'))
  .times(2)
  .dedup()
  .values('name')

// With path length tracking
g.V().has('id', 'alice')
  .emit()
  .repeat(out('TRANSFERS'))
  .times(2)
  .project('name', 'depth')
  .by(values('name'))
  .by(loops())
"""

print("Gremlin Query:")
print(gremlin_emit)

print("\nExpected Result:")
print("""
Alice        (starting node - emit)
Bob          (1 hop)
Charlie      (2 hops)
Frank        (2 hops)
Eve          (1 hop)
""")

print("\n💡 Emit Behavior:")
print("""
  .emit()                 → Emit current traverser
  .emit().repeat()        → Emit before repeating
  .repeat().emit()        → Emit after repeating
  
  With filter: .emit(has('risk','HIGH'))  → Emit only HIGH risk
""")

# Query 2.4: Termination - Money reaching specific target
print("\n\n" + "-" * 70)
print("Query 2.4: All Paths from Alice to David (Max 4 hops)")
print("-" * 70)

gremlin_targeted = """
// Find all paths from alice to david within 4 hops
g.V().has('id', 'alice')
  .repeat(out('TRANSFERS'))
  .until(has('id', 'david').or().loops().is(gte(4)))
  .filter(has('id', 'david'))
  .path()
  .by(values('name'))

// With edge amounts
g.V().has('id', 'alice')
  .repeat(out('TRANSFERS'))
  .until(has('id', 'david').or().loops().is(gte(4)))
  .filter(has('id', 'david'))
  .path()
  .by(project('node','amount')
      .by(values('name'))
      .by(optional(valueMap('amount'))))
"""

print("Gremlin Query:")
print(gremlin_targeted)

print("\nExpected Result:")
print("""
[Alice, Bob, Charlie, David]
[Alice, Bob, Frank, Alice, Bob, Charlie, David]
""")

print("\n💡 Termination Logic:")
print("""
  .until(condition)      → Stop when condition is TRUE
  .until(has('id','target'))  → Stop when we reach target
  .until(...).or().loops().is(gte(4))  → Stop at target OR 4 hops
  .filter()              → Final filter on result
""")

# Query 2.5: Aggregate amounts along path
print("\n\n" + "-" * 70)
print("Query 2.5: Total Amount Transferred (Alice to David)")
print("-" * 70)

gremlin_amounts = """
// Sum all transaction amounts from alice to david
g.V().has('id', 'alice')
  .repeat(outE('TRANSFERS').aggregate('edges'))
  .times(3)
  .filter(has('id', 'david'))
  .select('edges')
  .unfold()
  .values('amount')
  .sum()

// More practical: using path()
g.V().has('id', 'alice')
  .repeat(out('TRANSFERS'))
  .times(3)
  .filter(has('id', 'david'))
  .path()
  .map(unfold().filter(isEdge()).map(it.get().value('amount')).sum())
"""

print("Gremlin Query:")
print(gremlin_amounts)

print("\nExpected Result:")
print("""
8500  (Sum of amounts along path: 5000 + 2000 + 1500)
""")

print("\n✓ Query Type 2 covered: Multi-hop and loop control")

---

## 📊 Gremlin Query Type 3: Aggregation & Statistics

### Use Case: "Calculate network statistics per account"

**Key Gremlin Functions:**
- `.group()` - Group results by key
- `.sum()`, `.count()`, `.mean()` - Aggregation
- `.fold()` - Collect all into list
- `.unfold()` - Spread list back out

Unlike simple aggregation, Gremlin uses **group()** which is more flexible than GROUP BY in databases.

In [ ]:
# 🔹 Gremlin Query Type 3: Aggregation & Statistics

print("\n" + "=" * 70)
print("GREMLIN QUERY TYPE 3: AGGREGATION & STATISTICS")
print("=" * 70)

# Query 3.1: Transaction count per account
print("\n📌 Query 3.1: Transaction Statistics (Outgoing)")
print("-" * 70)

gremlin_stats = """
// Count outgoing transactions per account
g.V().hasLabel('Account')
  .project('account', 'outCount', 'outTotal', 'outAvg')
  .by(values('name'))
  .by(outE('TRANSFERS').count())
  .by(outE('TRANSFERS').values('amount').sum())
  .by(outE('TRANSFERS').values('amount').mean())
  .order().by(select('outTotal'), decr)

// Alternative using group()
g.V().outE('TRANSFERS')
  .group()
  .by(outV().values('name'))
  .by(project('count', 'total', 'avg')
    .by(count())
    .by(sum()))
  .unfold()
"""

print("Gremlin Query:")
print(gremlin_stats)

print("\nExpected Result:")
print("""
{
  "account": "Alice",
  "outCount": 2,
  "outTotal": 5500,
  "outAvg": 2750
}
{
  "account": "Bob",
  "outCount": 2,
  "outTotal": 3000,
  "outAvg": 1500
}
{
  "account": "Charlie",
  "outCount": 2,
  "outTotal": 3500,
  "outAvg": 1750
}
...
""")

print("\n💡 Aggregation Functions:")
print("""
  .count()           → Count items
  .sum()             → Sum values
  .mean()            → Average
  .min(), .max()     → Min/max
  .fold()            → Collect into list
  .group()           → Group by key
  .by()              → Apply to each
  .unfold()          → Spread list
""")

# Query 3.2: Incoming transactions
print("\n\n" + "-" * 70)
print("Query 3.2: Who Receives Money? (Incoming Stats)")
print("-" * 70)

gremlin_incoming = """
g.V().hasLabel('Account')
  .project('account', 'inCount', 'inTotal', 'inAvg')
  .by(values('name'))
  .by(inE('TRANSFERS').count())
  .by(inE('TRANSFERS').values('amount').sum())
  .by(inE('TRANSFERS').values('amount').mean())
  .order().by(select('inTotal'), decr)
"""

print("Gremlin Query:")
print(gremlin_incoming)

print("\nExpected Result:")
print("""
{"account":"Charlie", "inCount":3, "inTotal":5500, "inAvg":1833}
{"account":"Alice", "inCount":1, "inTotal":950, "inAvg":950}
{"account":"Bob", "inCount":1, "inTotal":5000, "inAvg":5000}
...
""")

print("\n🚨 Red Flags:")
print("  - Charlie: Receiver of $5500 from 3 accounts → HUB account")
print("  - Multiple incoming sources → Aggregation point")

# Query 3.3: Risk scoring
print("\n\n" + "-" * 70)
print("Query 3.3: Compute Risk Score (Degree + Volume)")
print("-" * 70)

gremlin_risk = """
g.V().hasLabel('Account')
  .project('account', 'baseRisk', 'outCount', 'inCount', 'degree',
           'outVolume', 'inVolume', 'computedRisk')
  .by(values('name'))
  .by(values('risk'))
  .by(outE('TRANSFERS').count())
  .by(inE('TRANSFERS').count())
  .by(coalesce(outE('TRANSFERS').count(), 0).plus(
      coalesce(inE('TRANSFERS').count(), 0)))
  .by(coalesce(outE('TRANSFERS').values('amount').sum(), 0))
  .by(coalesce(inE('TRANSFERS').values('amount').sum(), 0))
  .by(map(select('degree')
    .map(it -> it > 4 ? 'HIGH' : 
              it > 2 ? 'MEDIUM' : 'LOW')))
  .order().by(select('degree'), decr)
"""

print("Gremlin Query:")
print(gremlin_risk)

print("\nExpected Result:")
print("""
{
  "account": "Charlie",
  "baseRisk": "HIGH",
  "outCount": 2,
  "inCount": 3,
  "degree": 5,
  "outVolume": 3500,
  "inVolume": 5500,
  "computedRisk": "HIGH"
}
{
  "account": "Alice",
  "baseRisk": "LOW",
  "outCount": 2,
  "inCount": 1,
  "degree": 3,
  "outVolume": 5500,
  "inVolume": 950,
  "computedRisk": "MEDIUM"
}
...
""")

print("\n💡 Advanced Concepts:")
print("""
  .coalesce()        → Return first non-null value
  .plus()            → Addition
  .map()             → Transform with lambda
  it                 → Current object in lambda
  it > value ? yes : no  → Ternary operator
""")

# Query 3.4: Daily stats (temporal)
print("\n\n" + "-" * 70)
print("Query 3.4: Group by Date - Daily Activity")
print("-" * 70)

gremlin_daily = """
g.E('TRANSFERS')
  .group()
  .by(values('date'))
  .by(project('count', 'total', 'min', 'max')
    .by(count())
    .by(values('amount').sum())
    .by(values('amount').min())
    .by(values('amount').max()))
  .unfold()
  .order().by(keys)
"""

print("Gremlin Query:")
print(gremlin_daily)

print("\nExpected Result:")
print("""
[
  ["2024-01-15", {"count": 1, "total": 5000, "min": 5000, "max": 5000}],
  ["2024-01-20", {"count": 1, "total": 2000, "min": 2000, "max": 2000}],
  ...
]
""")

print("\n✓ Query Type 3 covered: Aggregation and statistics")

---

## 🔄 Gremlin Query Type 4: Cycle Detection (Critical for Fraud)

### Use Case: "Find money that returns to source (circular routing)"

**Gremlin Pattern:**
```groovy
g.V().repeat(...).until(same as starting node)
```

This is where Gremlin's loop control and `repeat().until()` really shine for detecting cycles.

In [ ]:
# 🔹 Gremlin Query Type 4: Cycle Detection

print("\n" + "=" * 70)
print("GREMLIN QUERY TYPE 4: CYCLE DETECTION")
print("=" * 70)

# Query 4.1: Find cycles
print("\n📌 Query 4.1: Find Cycles (Money Returning to Source)")
print("-" * 70)

gremlin_cycles = """
// Find accounts that are part of cycles (2-5 hops back to self)
g.V().has('id', 'alice')
  .repeat(out('TRANSFERS'))
  .until(has('id', 'alice').or().loops().is(gte(5)))
  .filter(has('id', 'alice'))
  .path()
  .by(values('name'))

// Find ALL accounts in ANY cycle
g.V()
  .as('start')
  .repeat(out('TRANSFERS'))
  .until(sack().by(identity()).is(eq(select('start'))).or().loops().is(gte(5)))
  .filter(sack().by(identity()).is(eq(select('start'))))
  .select('start')
  .dedup()
  .values('name')
"""

print("Gremlin Query:")
print(gremlin_cycles)

print("\nExpected Result:")
print("""
[Alice, Bob, Frank, Alice]

With all cycles:
Alice
(any other accounts in cycles)
""")

print("\n💡 Cycle Detection Techniques:")
print("""
  .as('start')           → Name the starting vertex
  .select('start')       → Reference it later
  .filter(eq(...))       → Filter where we return to start
  .sack()                → State accumulator (advanced)
  
  Simple approach: Just check if path ends at start node
  Advanced: Use sack() for accumulating cycle information
""")

# Query 4.2: High-value cycles
print("\n\n" + "-" * 70)
print("Query 4.2: Cycles with High Total Amount (> $5000)")
print("-" * 70)

gremlin_high_cycles = """
g.V().hasLabel('Account')
  .as('start')
  .repeat(outE('TRANSFERS').aggregate('edges').outV())
  .until(has('id', select('start')).by(values('id'))
    .or().loops().is(gte(5)))
  .filter(has('id', select('start')).by(values('id')))
  .select('edges')
  .map(unfold().values('amount').sum())
  .is(gt(5000))
  .project('account', 'cycleAmount')
  .by(select('start').by(values('name')))
  .by(select('edges')
    .map(unfold().values('amount').sum()))
"""

print("Gremlin Query:")
print(gremlin_high_cycles)

print("\nExpected Result:")
print("""
{"account": "Alice", "cycleAmount": 6950}
""")

print("\n🚨 Fraud Alert:")
print("  - Alice is part of cycle with $6950 total → SUSPICIOUS")

# Query 4.3: Triangle patterns (tightest cycles)
print("\n\n" + "-" * 70)
print("Query 4.3: Triangle Patterns (3-Node Cycles)")
print("-" * 70)

gremlin_triangles = """
// Find 3-node cycles
g.V()
  .as('a')
  .out('TRANSFERS')
  .as('b')
  .out('TRANSFERS')
  .as('c')
  .out('TRANSFERS')
  .filter(has('id', select('a').by(values('id'))))
  .project('node1', 'node2', 'node3',
           'amount1', 'amount2', 'amount3')
  .by(select('a').by(values('name')))
  .by(select('b').by(values('name')))
  .by(select('c').by(values('name')))
  .by(select('a').by(outE('TRANSFERS').filter(inV().has('id', 
      select('b').by(values('id')))).values('amount')))
  .by(select('b').by(outE('TRANSFERS').filter(inV().has('id', 
      select('c').by(values('id')))).values('amount')))
  .by(select('c').by(outE('TRANSFERS').filter(inV().has('id', 
      select('a').by(values('id')))).values('amount')))
"""

print("\nGremlin Query (Complex Triangle Detection):")
print(gremlin_triangles)

print("\nExpected Result:")
print("""
(No results - no 3-node cycles in our example network)
""")

print("\n✓ Query Type 4 covered: Cycle detection")

---

## ⭐ Gremlin Query Type 5: Hub Detection (Star Patterns)

### Use Case: "Find money routing centers"

**Gremlin Hub Pattern:**
```groovy
g.V().as('hub')
  .both('TRANSFERS')  // Both incoming and outgoing
  .dedup()
  .count()  // Count unique neighbors
```

In [ ]:
# 🔹 Gremlin Query Type 5: Hub Detection

print("\n" + "=" * 70)
print("GREMLIN QUERY TYPE 5: HUB DETECTION")
print("=" * 70)

# Query 5.1: Find hub accounts
print("\n📌 Query 5.1: Identify Hub Accounts (High Connectivity)")
print("-" * 70)

gremlin_hubs = """
// Find accounts with high degree
g.V().hasLabel('Account')
  .as('account')
  .project('name', 'outDegree', 'inDegree', 'totalDegree',
           'outVolume', 'inVolume')
  .by(values('name'))
  .by(outE('TRANSFERS').count())
  .by(inE('TRANSFERS').count())
  .by(coalesce(outE('TRANSFERS').count(),0)
    .plus(coalesce(inE('TRANSFERS').count(),0)))
  .by(coalesce(outE('TRANSFERS').values('amount').sum(),0))
  .by(coalesce(inE('TRANSFERS').values('amount').sum(),0))
  .order().by(select('totalDegree'), decr)
"""

print("Gremlin Query:")
print(gremlin_hubs)

print("\nExpected Result:")
print("""
{"name": "Charlie", "outDegree": 2, "inDegree": 3, "totalDegree": 5,
 "outVolume": 3500, "inVolume": 5500}
{"name": "Alice", "outDegree": 2, "inDegree": 1, "totalDegree": 3,
 "outVolume": 5500, "inVolume": 950}
{"name": "Bob", "outDegree": 2, "inDegree": 1, "totalDegree": 3,
 "outVolume": 3000, "inVolume": 5000}
...
""")

print("\n💡 Hub Scoring:")
print("""
  Charlie (degree 5): Primary hub
  Alice (degree 3, high out-volume): Source node
  Bob (degree 3): Middle player
  
  Red flag: High inflow + lower outflow (consolidation)
""")

# Query 5.2: Hub star pattern
print("\n\n" + "-" * 70)
print("Query 5.2: Star Pattern - All Connections Around Hub")
print("-" * 70)

gremlin_star = """
// Get all neighbors of charlie (hub)
g.V().has('name', 'charlie')
  .as('hub')
  .bothE('TRANSFERS')
  .project('direction', 'amount', 'otherParty')
  .by(direction)
  .by(values('amount'))
  .by(otherV().values('name'))
  .order().by(select('amount'), decr)

// Alternative: Separate in/out
g.V().has('name', 'charlie')
  .project('senders', 'receivers')
  .by(in('TRANSFERS').values('name').fold())
  .by(out('TRANSFERS').values('name').fold())
"""

print("Gremlin Query:")
print(gremlin_star)

print("\nExpected Result:")
print("""
Direction: FORWARD
[{"direction": "OUT", "amount": 2000, "otherParty": "Eve"},
 {"direction": "OUT", "amount": 1500, "otherParty": "David"},
 {"direction": "IN", "amount": 3000, "otherParty": "David"},
 {"direction": "IN", "amount": 2000, "otherParty": "Bob"},
 {"direction": "IN", "amount": 5000, "otherParty": "Alice"}]
""")

print("\n✓ Query Type 5 covered: Hub detection")

---

## 🎯 Gremlin Query Type 6: Advanced Investigation Queries

### Use Case: "Comprehensive risk assessment combining multiple factors"

**Neptune Analytics Integration:**
- Built-in algorithms (PageRank, Louvain) accessible via Gremlin
- Combine traversal results with algorithm outputs
- Real-time scoring for transaction processing

In [ ]:
# 🔹 Gremlin Query Type 6: Advanced Investigation & Risk Scoring

print("\n" + "=" * 70)
print("GREMLIN QUERY TYPE 6: ADVANCED INVESTIGATION")
print("=" * 70)

# Query 6.1: Multi-factor risk assessment
print("\n📌 Query 6.1: Comprehensive Risk Score")
print("-" * 70)

gremlin_advanced = """
// Multi-factor risk scoring for all accounts
g.V().hasLabel('Account')
  .project('account', 'baseRisk', 'degreeScore', 'volumeScore',
           'cycleScore', 'finalRisk')
  .by(values('name'))
  .by(values('risk'))
  .by(map(
    select('account').as('a')
    .by(coalesce(outE('TRANSFERS').count(), 0)
      .plus(coalesce(inE('TRANSFERS').count(), 0)))
    .map(it > 5 ? 3.0 : it > 2 ? 1.5 : 0.5)))
  .by(map(
    select('account')
    .by(coalesce(outE('TRANSFERS').values('amount').sum(), 0)
      .plus(coalesce(inE('TRANSFERS').values('amount').sum(), 0)))
    .map(it > 10000 ? 2.0 : it > 5000 ? 1.0 : 0.5)))
  .by(map(
    select('account')
    .as('start')
    .repeat(out('TRANSFERS'))
    .until(has('id', select('start').by(values('id')))
      .or().loops().is(gte(3)))
    .filter(has('id', select('start').by(values('id'))))
    .count()
    .map(it > 0 ? 2.0 : 0)))
  .by(map(
    (select('degreeScore') + select('volumeScore') + 
     select('cycleScore')) / 3.0
    .map(it > 1.5 ? 'HIGH' : it > 0.8 ? 'MEDIUM' : 'LOW')))
  .order().by(select('finalRisk'), decr)
"""

print("Gremlin Query:")
print(gremlin_advanced)

print("\nExpected Result:")
print("""
{
  "account": "Charlie",
  "baseRisk": "HIGH",
  "degreeScore": 1.5,
  "volumeScore": 1.0,
  "cycleScore": 0,
  "finalRisk": "HIGH"
}
{
  "account": "Alice",
  "baseRisk": "LOW",
  "degreeScore": 1.5,
  "volumeScore": 1.0,
  "cycleScore": 2.0,
  "finalRisk": "MEDIUM"
}
...
""")

print("\n💡 Risk Factors:")
print("""
  Degree Score    → High connectivity (many connections)
  Volume Score    → High transaction volumes
  Cycle Score     → Participation in money cycles
  Final Risk      → Weighted average of all factors
""")

# Query 6.2: Suspicious pattern detection
print("\n\n" + "-" * 70)
print("Query 6.2: Suspicious Patterns (Rapid Movement)")
print("-" * 70)

gremlin_patterns = """
// Find rapid 3-hop transfers (within same day or consecutive)
g.E('TRANSFERS')
  .has('date', 'gte:2024-01-15')
  .has('date', 'lte:2024-02-15')
  .outV()
  .as('start')
  .repeat(out('TRANSFERS'))
  .times(2)
  .path()
  .by(project('node','date','amount')
    .by(values('name'))
    .by(optional(values('date')))
    .by(optional(values('amount'))))
  .filter(select(last).values('name')
    .is(neq(select('start').by(values('name')))))
  .order()
  .limit(10)
"""

print("Gremlin Query:")
print(gremlin_patterns)

print("\nExpected Result:")
print("""
Rapid paths showing money movement:
[Alice, Bob, Charlie, David]
[Alice, Bob, Frank, ...]
...
""")

# Query 6.3: Investigation - Which accounts to watch
print("\n\n" + "-" * 70)
print("Query 6.3: Watchlist - Accounts Requiring Investigation")
print("-" * 70)

gremlin_watchlist = """
g.V().hasLabel('Account')
  .filter(
    coalesce(outE('TRANSFERS').count(), 0)
      .plus(coalesce(inE('TRANSFERS').count(), 0)).is(gte(3))
    .or()
    coalesce(outE('TRANSFERS').values('amount').sum(), 0)
      .plus(coalesce(inE('TRANSFERS').values('amount').sum(), 0))
      .is(gte(5000))
    .or()
    has('risk', 'HIGH'))
  .project('account', 'connections', 'volume', 'risk', 'reasons')
  .by(values('name'))
  .by(coalesce(outE('TRANSFERS').count(), 0)
    .plus(coalesce(inE('TRANSFERS').count(), 0)))
  .by(coalesce(outE('TRANSFERS').values('amount').sum(), 0)
    .plus(coalesce(inE('TRANSFERS').values('amount').sum(), 0)))
  .by(values('risk'))
  .by(project('reasons')
    .by(fold()))  // Simplified - real version would build reason list
"""

print("Gremlin Query:")
print(gremlin_watchlist)

print("\nExpected Result:")
print("""
{
  "account": "Charlie",
  "connections": 5,
  "volume": 9000,
  "risk": "HIGH",
  "reasons": ["High connectivity", "High volume", "HIGH base risk"]
}
{
  "account": "Alice",
  "connections": 3,
  "volume": 6450,
  "risk": "LOW",
  "reasons": ["High volume", "Part of cycle"]
}
...
""")

print("\n✓ Query Type 6 covered: Advanced investigation")

---

## 🏗️ Neptune + AWS Architecture for Production Fraud Detection

### Neptune Architecture (5-Layer Model)

```
┌─────────────────────────────────────────────────┐
│  APPLICATION LAYER                              │
│  - Real-time Transaction Processing             │
│  - SageMaker ML Models                          │
│  - Lambda Functions                             │
│  - API Gateway                                  │
└─────────────────────────────────────────────────┘
                      ↓
┌─────────────────────────────────────────────────┐
│  QUERY EXECUTION LAYER                          │
│  - Gremlin Query Engine                         │
│  - Query Optimization                           │
│  - Result Caching                               │
└─────────────────────────────────────────────────┘
                      ↓
┌─────────────────────────────────────────────────┐
│  ANALYTICS LAYER                                │
│  - Neptune Analytics (PageRank, Louvain)        │
│  - Graph Algorithms                             │
│  - Batch Processing                             │
└─────────────────────────────────────────────────┘
                      ↓
┌─────────────────────────────────────────────────┐
│  STORAGE LAYER                                  │
│  - Multi-AZ Replication                         │
│  - Snapshots to S3                              │
│  - Read Replicas                                │
└─────────────────────────────────────────────────┘
                      ↓
┌─────────────────────────────────────────────────┐
│  AWS INTEGRATION LAYER                          │
│  - S3 (Data Import/Export)                      │
│  - Kinesis/Kafka (Streaming)                    │
│  - Lambda (Triggers)                            │
│  - CloudWatch (Monitoring)                      │
│  - SageMaker (ML)                               │
└─────────────────────────────────────────────────┘
```

### Data Ingestion Pipeline

```
Transaction Stream (Kafka/Kinesis)
        ↓
Lambda Function (every 100 txns or 30s)
        ↓
Add to Neptune (Gremlin)
        ↓
Run Analysis Queries
        ↓
Score & Alert
```

### Real-Time vs Batch Processing

**Real-Time (Gremlin):**
- Incoming transaction triggers Lambda
- Query: Check for cycles, high-degree nodes
- Latency: < 100ms per query
- Throughput: ~1000 txns/sec per Neptune DB

**Batch (Neptune Analytics):**
- Daily analysis of patterns
- Louvain community detection
- PageRank importance scoring
- Latency: Minutes for full graph analysis
- Cost-effective for 1M+ nodes

In [ ]:
# 🏗️ Neptune + SageMaker Integration

print("\n" + "=" * 70)
print("NEPTUNE + SAGEMAKER INTEGRATION ARCHITECTURE")
print("=" * 70)

integration_code = """
# Step 1: Extract Features from Neptune using Gremlin
# ====================================================
import boto3
from gremlin_python.process.graph_traversal import traversal
from gremlin_python.driver.driver_remote_connection import DriverRemoteConnection

# Connect to Neptune
conn = DriverRemoteConnection('wss://neptune-db.amazonaws.com:8182/gremlin', 'g')
g = traversal().withRemote(conn)

# Extract features for ML
def extract_account_features(account_id):
    # Query 1: Basic stats
    out_degree = g.V().has('id', account_id).outE('TRANSFERS').count().next()
    in_degree = g.V().has('id', account_id).inE('TRANSFERS').count().next()
    
    # Query 2: Volume
    out_volume = g.V().has('id', account_id)\
        .outE('TRANSFERS').values('amount').sum().next() or 0
    in_volume = g.V().has('id', account_id)\
        .inE('TRANSFERS').values('amount').sum().next() or 0
    
    # Query 3: Cycle participation
    cycles = g.V().has('id', account_id).as('start')\
        .repeat(out('TRANSFERS'))\
        .until(has('id', account_id).or().loops().is(gte(4)))\
        .filter(has('id', account_id)).count().next() or 0
    
    return {
        'out_degree': out_degree,
        'in_degree': in_degree,
        'out_volume': out_volume,
        'in_volume': in_volume,
        'cycle_count': cycles,
        'total_degree': out_degree + in_degree,
        'in_out_ratio': in_volume / (out_volume + 0.01)  # Avoid div by 0
    }

# Step 2: Export to S3 & SageMaker
# ================================
import pandas as pd
from sagemaker import Session

session = Session()
bucket = session.default_bucket()

# Collect features for all accounts
account_ids = ['alice', 'bob', 'charlie', 'david', 'eve', 'frank']
features_list = [extract_account_features(aid) for aid in account_ids]
df = pd.DataFrame(features_list)

# Save to S3
csv_path = f"s3://{bucket}/fraud-detection/features.csv"
df.to_csv(csv_path, index=False)

# Step 3: Train SageMaker Model
# =============================
from sagemaker.xgboost.estimator import XGBoost

xgb_estimator = XGBoost(
    entry_point='train.py',
    role='arn:aws:iam::ACCOUNT:role/SageMakerRole',
    instance_count=1,
    instance_type='ml.m5.large',
    framework_version='1.5-1',
    py_version='python3',
    hyperparameters={
        'max_depth': 5,
        'eta': 0.2,
        'objective': 'binary:logistic',
        'num_round': 100
    }
)

xgb_estimator.fit(csv_path)

# Step 4: Deploy to Real-Time Endpoint
# ====================================
predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium'
)

# Step 5: Real-Time Scoring Lambda
# =================================
def lambda_handler(event, context):
    '''
    Triggered when new transaction is added to Neptune
    '''
    account_id = event['account_id']
    
    # Extract features from Neptune
    features = extract_account_features(account_id)
    
    # Call SageMaker endpoint
    runtime = boto3.client('sagemaker-runtime')
    response = runtime.invoke_endpoint(
        EndpointName='fraud-detection-endpoint',
        ContentType='text/csv',
        Body=','.join(str(v) for v in features.values())
    )
    
    fraud_probability = float(response['Body'].read().decode())
    
    if fraud_probability > 0.7:
        # Alert & block
        alert_sns(account_id, fraud_probability)
    
    return {'statusCode': 200, 'fraud_score': fraud_probability}

# Step 6: Batch Retraining (Daily)
# ================================
def batch_retrain():
    '''
    Daily job to retrain model with new patterns
    '''
    # 1. Get all accounts with labels
    # 2. Extract features from Neptune
    # 3. Combine with manual fraud labels from investigators
    # 4. Save to S3
    # 5. Trigger SageMaker retraining job
    pass
"""

print("\nIntegration Code Example:")
print("-" * 70)
print(integration_code)

print("\n\n" + "=" * 70)
print("KEY FEATURES OF NEPTUNE + SAGEMAKER")
print("=" * 70)

features = """
1. FEATURE ENGINEERING
   - Neptune Gremlin queries provide real-time graph features
   - Degree, centrality, clustering coefficients
   - Cycle participation, path length statistics
   - Community membership (from Louvain algorithm)

2. REAL-TIME SCORING
   - New transaction → Lambda trigger
   - Extract features from Neptune (< 100ms)
   - Call SageMaker endpoint (< 50ms)
   - Total latency: < 200ms
   - Decision: Block, Allow, or Review

3. FEEDBACK LOOP
   - Investigators label suspected fraud
   - Labels stored in Neptune
   - Daily retraining with new patterns
   - Model continuously improves

4. SCALABILITY
   - Neptune handles 1B+ nodes
   - SageMaker auto-scales endpoints
   - Lambda concurrent executions
   - No single bottleneck

5. COST OPTIMIZATION
   - Pay per write to Neptune
   - On-demand SageMaker endpoints
   - Lambda per-invocation pricing
   - Reserved capacity for predictable load

6. MONITORING & ALERTING
   - CloudWatch dashboards
   - Model drift detection
   - Query performance tracking
   - Alert to security team on anomalies
"""

print(features)

print("\n" + "=" * 70)
print("CYPHER vs GREMLIN: WHEN TO USE EACH")
print("=" * 70)

comparison = """
USE CYPHER (Neo4j) WHEN:
✓ Complex pattern matching is primary need
✓ ACID transactions required
✓ Smaller dataset (< 100M nodes)
✓ Team familiar with SQL-like syntax
✓ Need sophisticated JOIN semantics

USE GREMLIN (Neptune) WHEN:
✓ Real-time streaming requirements
✓ Massive scale (1B+ nodes)
✓ Need AWS ecosystem integration
✓ Multiple query languages (Cypher + Gremlin)
✓ Cost-sensitive (pay per write, not per query)
✓ Require analytics + traversal combined

IN FRAUD DETECTION:
Neptune/Gremlin typically better because:
- High transaction volume requires streaming
- Must scale to billions of relationships
- SageMaker integration critical
- AWS Lambda triggers for real-time
- Analytics (PageRank) needed for insight
"""

print(comparison)

print("\n✓ Neptune + SageMaker architecture documented")

---

## 📚 Enhanced Summary: Gremlin Reference & Production Guide

### 1. Gremlin Quick Reference (All Query Types)

| Query Type | Use Case | Gremlin Pattern | Complexity |
|-----------|----------|-----------------|-----------|
| **Simple Traversal** | Direct connections | `g.V().has('id','x').out('REL').values('prop')` | O(E) |
| **Multi-Hop** | Paths of length N | `g.V().has('id','x').repeat(out('REL')).times(N)` | O(E^N) |
| **Aggregation** | Stats per account | `g.V().project(...).by(count()).by(sum())` | O(V) |
| **Cycle Detection** | Money loops | `g.V().as('start').repeat(out('REL')).until(eq('start'))` | O(E) |
| **Hub Detection** | High degree nodes | `g.V().project('degree').by(bothE().count())` | O(V) |
| **Investigation** | Multi-factor scoring | `g.V().project(...).by(aggregations)` | O(V + E) |

### 2. Gremlin Methods Cheat Sheet

**Traversal Starting Points:**
- `g.V()` - All vertices
- `g.V(id1, id2)` - Specific vertices by ID
- `g.E()` - All edges
- `g.V().hasLabel('Account')` - Filter by label

**Navigation:**
- `.out('TRANSFERS')` - Outgoing edges
- `.in('TRANSFERS')` - Incoming edges
- `.both('TRANSFERS')` - Both directions
- `.outV()` / `.inV()` - Destination vertex
- `.bothV()` - Both vertices of edge

**Filtering:**
- `.has('property', value)` - Property match
- `.has('property', gt(1000))` - Comparison: gt, lt, gte, lte, eq, neq
- `.has('property', inside(min, max))` - Range
- `.filter(condition)` - Custom filter

**Aggregation:**
- `.count()` - Count items
- `.sum()` - Sum values
- `.mean()` / `.min()` / `.max()` - Statistics
- `.group().by(key)` - Group by property
- `.fold()` - Collect into list
- `.unfold()` - Spread list items

**Transformation:**
- `.project('col1','col2')` - Select columns
- `.by(expression)` - Apply to column
- `.map(lambda)` - Transform each item
- `.values('prop')` - Extract property
- `.valueMap()` - All properties as map
- `.path()` - Capture path taken

**Loop Control:**
- `.repeat(step)` - Loop pattern
- `.times(N)` - Exactly N iterations
- `.until(condition)` - Loop until condition
- `.emit()` - Output intermediate results
- `.loops()` - Current iteration count

**Ordering:**
- `.order()` - Sort results
- `.by(select('field'))` - Sort key
- `.by(..., decr)` - Descending order
- `.limit(N)` - Top N results

**Deduplication:**
- `.dedup()` - Remove duplicates
- `.dedup().by('field')` - Dedup by field

### 3. Neptune Performance Tuning

**Query Optimization:**

```gremlin
// ❌ SLOW: Traverse everything then filter
g.V().out('TRANSFERS').filter(has('amount', gt(1000)))

// ✅ FAST: Filter before traversing
g.V().outE('TRANSFERS').has('amount', gt(1000)).inV()

// ❌ SLOW: Get all properties
g.V().hasLabel('Account').valueMap()

// ✅ FAST: Get only needed properties
g.V().hasLabel('Account').project('id', 'name').by(values('id')).by(values('name'))

// ❌ SLOW: Multiple queries
amount1 = g.E().has('date','2024-01-15').values('amount').sum()
amount2 = g.E().has('date','2024-01-16').values('amount').sum()

// ✅ FAST: Single query grouped
g.E().group().by(values('date')).by(values('amount').sum())
```

**Indexing Strategy:**

```gremlin
// Create indexes for common filters (do once)
// - On 'id' property (lookups)
// - On 'date' property (temporal filters)
// - On 'risk' property (classification)
// - On 'amount' property (range filters)

// Query with indexed property is O(log E) instead of O(E)
g.V().has('id', 'alice')  // Fast (indexed)
g.V().has('name', 'Alice')  // Slow (no index)
```

**Batch Processing:**

```
// Instead of per-record queries:
❌ 1000 individual transactions → 1000 separate Gremlin queries
   Latency: 1000 * 100ms = 100 seconds

// Use batch:
✅ 1000 transactions in batch → 1 bulk load operation
   Latency: 5 seconds
```

### 4. Real-World Fraud Detection Pipeline

```
TRANSACTION FLOW:
═══════════════════════════════════════════════════════════

1. INGESTION
   Transaction → Kafka/Kinesis → Batch (every 100 or 30s)

2. ENRICHMENT
   Add to Neptune + Extract features via Gremlin queries
   - Degree, volume, patterns
   - Cycle detection
   - Community membership

3. SCORING
   Call SageMaker ML model with features
   Score: 0.0 (safe) → 1.0 (fraud)

4. DECISION
   Score > 0.9: BLOCK (immediate)
   Score > 0.7: REVIEW (analyst queue)
   Score < 0.7: ALLOW (log for analysis)

5. FEEDBACK
   Investigators manually review flagged transactions
   Labels saved back to Neptune
   Daily retraining with new patterns

6. MONITORING
   - CloudWatch: Query latency, error rates
   - Model: Accuracy, precision, recall
   - Graph: Growth rate, clustering coefficient
```

### 5. Deployment Checklist

- [ ] Neptune cluster configured (multi-AZ)
- [ ] SageMaker endpoint deployed
- [ ] Lambda functions for real-time scoring
- [ ] CloudWatch alarms for anomalies
- [ ] S3 bucket for backups & exports
- [ ] Kinesis/Kafka topic configured
- [ ] IAM roles & permissions set
- [ ] Data privacy (encryption, audit logging)
- [ ] Automated backups to S3 (daily)
- [ ] Disaster recovery plan documented
- [ ] Query performance baselines measured
- [ ] Cost monitoring alerts set

### 6. Gremlin vs Cypher vs SQL Comparison

```python
TASK: Find all accounts that sent money to Bob

SQL:
─────
SELECT DISTINCT sender_id 
FROM transactions 
WHERE receiver_id = 'bob'

Cypher (Neo4j):
───────────────
MATCH (sender)-[:TRANSFERS]->(receiver:Account {id:'bob'})
RETURN DISTINCT sender.id

Gremlin (Neptune):
──────────────────
g.V().has('id','bob')
  .in('TRANSFERS')
  .dedup()
  .values('id')
```

### 7. Common Pitfalls & Solutions

| Problem | Cause | Solution |
|---------|-------|----------|
| Query timeout | Too many traversals | Add `.limit()`, optimize indexes |
| Memory error | `.fold()` on large result | Use `.fold().unfold()` or stream |
| Wrong results | Forgot `.dedup()` | Always deduplicate paths |
| Slow cycles | O(E^N) complexity | Limit `.times()` or add `.until()` |
| Data inconsistency | Read replicas lag | Use write instance for critical reads |

### 8. Interview Questions

1. **Q: What's the difference between `.repeat()` and `.until()` in Gremlin?**
   A: `.repeat()` defines the pattern; `.until()` defines termination condition. Combined they loop until condition is true.

2. **Q: Why use Neptune over Neo4j for fraud detection?**
   A: Scale (1B+ nodes), AWS integration (SageMaker, Lambda), streaming (Kinesis), cost model (pay per write).

3. **Q: How would you detect money laundering in a large graph?**
   A: Multi-factor scoring: degree + cycles + clustering coefficient + community detection + temporal patterns.

4. **Q: What's the complexity of detecting a 5-hop cycle?**
   A: O(E^5) worst case, but with good indexing and early termination, typically O(E^2).

5. **Q: How do you handle real-time scoring at high transaction volume?**
   A: Batch ingestion + caching + SageMaker endpoints auto-scaling + Lambda concurrency limits.

# 🧩 Summary

* Neptune enables **large-scale graph analytics**
* Supports **Gremlin & SPARQL**
* Used for **fraud detection, AML, recommendations**
* Integrates with AWS ecosystem
* Critical for production systems

---

### ❓ Interview Questions

* Difference between Neo4j and Neptune?
* Why use Gremlin instead of SQL?
* How do you scale graph analytics?
* How does Neptune integrate with ML pipelines?

---

### 🧪 Mini Exercise Idea

* Write Gremlin queries to:

  * Find neighbors
  * Find 2-hop connections
  * Simulate fraud tracing